In [2]:
import torch
from scipy.stats import norm


In [11]:
def make_nf4_codebook():
    """
    Build the 16-value NF4 codebook: quantiles of a standard normal
    distribution, asymmetric (8 negative, exact 0, 7 positive) so that
    zero is exactly representable.
    """
    offset = 0.9677083
    neg_probs = torch.linspace(1 - offset, 0.5, 9)[:-1]
    neg = norm.ppf(neg_probs.tolist())
    pos_probs = torch.linspace(0.5, offset, 8)[1:]
    pos = norm.ppf(pos_probs.tolist())
    codebook = torch.tensor(list(neg) + [0.0] + list(pos), dtype=torch.float32)
    codebook = torch.sort(codebook).values
    codebook = codebook / codebook.abs().max()
    return codebook


In [4]:
def quantize_nf4(tensor: torch.Tensor, codebook: torch.Tensor, block_size: int = 64):
    """
    Quantize a float tensor to 4-bit NF4 indices, block-wise.
    Returns: indices (uint8), scales (per-block absmax), orig_shape, orig_numel.
    """
    flat = tensor.flatten()
    n = flat.numel()
    pad = (-n) % block_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad)])
    blocks = flat.view(-1, block_size)

    scales = blocks.abs().max(dim=1, keepdim=True).values
    scales = scales.clamp(min=1e-8)
    normalized = blocks / scales

    diffs = (normalized.unsqueeze(-1) - codebook.view(1, 1, -1)).abs()
    indices = diffs.argmin(dim=-1).to(torch.uint8)

    return indices, scales.squeeze(1), tensor.shape, n

In [ ]:
def dequantize_nf4(indices: torch.Tensor, scales: torch.Tensor, codebook: torch.Tensor, orig_shape, orig_numel: int, block_size: int = 64):
    """Reverse of quantize_nf4: index lookup, then rescale by the block's absmax."""
    looked_up = codebook[indices.long()]
    dequantized = looked_up * scales.unsqueeze(1)
    flat = dequantized.flatten()[:orig_numel]
    return flat.view(orig_shape)

In [8]:
def naive_int4_quantize(tensor, block_size=64):
    flat = tensor.flatten()
    n = flat.numel()
    pad = (-n) % block_size
    if pad:
        flat = torch.cat([flat, torch.zeros(pad)])
    blocks = flat.view(-1, block_size)
    scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normalized = blocks / scales

    levels = torch.linspace(-1, 1, 16)   # <-- the ONLY real difference from NF4
    diffs = (normalized.unsqueeze(-1) - levels.view(1, 1, -1)).abs()
    idx = diffs.argmin(dim=-1)

    dequant = levels[idx] * scales
    return dequant.flatten()[:n].view(tensor.shape)

In [12]:
if __name__ == "__main__":
    cb = make_nf4_codebook()
    print("Codebook:", cb)
 
    # --- small readable test (8x4, block_size=8) ---
    torch.manual_seed(0)
    test_tensor = torch.randn(4, 8) * 0.02
 
    indices, scales, shape, n = quantize_nf4(test_tensor, cb, block_size=8)
    reconstructed = dequantize_nf4(indices, scales, cb, shape, n, block_size=8)
 
    print("\n--- Small test tensor (4x8, block_size=8) ---")
    print("Original:\n", test_tensor)
    print("Indices:\n", indices)
    print("Scales:\n", scales)
    print("Reconstructed:\n", reconstructed)
    print("Max absolute error:", (test_tensor - reconstructed).abs().max().item())
 
    naive_reconstructed = naive_int4_quantize(test_tensor, block_size=8)
    nf4_mse_small = torch.mean((test_tensor - reconstructed) ** 2).item()
    int4_mse_small = torch.mean((test_tensor - naive_reconstructed) ** 2).item()
    print(f"\n[Small tensor, N=32 -- too few samples for the statistical NF4")
    print(f" advantage to show reliably, kept here only for illustration]")
    print(f"NF4 MSE: {nf4_mse_small:.10f}   Naive INT4 MSE: {int4_mse_small:.10f}")
 
    # --- large realistic test (1024x1024, block_size=64) ---
    torch.manual_seed(0)
    big_tensor = torch.randn(1024, 1024) * 0.02  # ~1M values, weight-matrix-like scale
 
    indices, scales, shape, n = quantize_nf4(big_tensor, cb, block_size=64)
    nf4_recon = dequantize_nf4(indices, scales, cb, shape, n, block_size=64)
    int4_recon = naive_int4_quantize(big_tensor, block_size=64)
 
    nf4_mse = torch.mean((big_tensor - nf4_recon) ** 2).item()
    int4_mse = torch.mean((big_tensor - int4_recon) ** 2).item()
 
    print("\n--- Large tensor (1024x1024 = 1,048,576 values, block_size=64) ---")
    print("Original (top-left 8x8 slice):\n", big_tensor[:8, :8])
    print("\nNF4 reconstructed (same slice):\n", nf4_recon[:8, :8])
    print("\nNaive INT4 reconstructed (same slice):\n", int4_recon[:8, :8])
 
    print(f"\n--- Full tensor stats (all {big_tensor.numel()} values) ---")
    print(f"NF4 MSE:        {nf4_mse:.10f}")
    print(f"Naive INT4 MSE: {int4_mse:.10f}")
    print(f"NF4 improvement: {(1 - nf4_mse/int4_mse)*100:.2f}%")
 
    # --- compression ratio check ---
    orig_bytes = big_tensor.numel() * 4  # fp32
    packed_bytes = (big_tensor.numel() * 4 // 8) + (scales.numel() * 4)  # 4-bit/val + fp32 scales
    print(f"\nOriginal (fp32): {orig_bytes/1e6:.2f} MB")
    print(f"NF4 packed (4-bit/val + scales): {packed_bytes/1e6:.2f} MB")
    print(f"Compression ratio: {orig_bytes/packed_bytes:.2f}x")
 

Codebook: tensor([-1.0000, -0.7230, -0.5626, -0.4407, -0.3379, -0.2461, -0.1609, -0.0796,
         0.0000,  0.0911,  0.1848,  0.2844,  0.3949,  0.5251,  0.6962,  1.0000])

--- Small test tensor (4x8, block_size=8) ---
Original:
 tensor([[-0.0225, -0.0230, -0.0050, -0.0087,  0.0170,  0.0138, -0.0063, -0.0423],
        [ 0.0064, -0.0253,  0.0070,  0.0062,  0.0024,  0.0248,  0.0223, -0.0049],
        [-0.0271, -0.0339,  0.0113,  0.0159,  0.0120, -0.0311, -0.0068,  0.0371],
        [ 0.0150, -0.0117, -0.0035,  0.0037,  0.0278,  0.0317,  0.0189, -0.0169]])
Indices:
 tensor([[ 2,  2,  7,  5, 12, 11,  6,  0],
        [11,  0, 11, 11,  9, 15, 15,  6],
        [ 1,  0, 11, 12, 11,  1,  6, 15],
        [13,  4,  7,  9, 15, 15, 13,  2]], dtype=torch.uint8)
Scales:
 tensor([0.0423, 0.0253, 0.0371, 0.0317])
Reconstructed:
 tensor([[-0.0238, -0.0238, -0.0034, -0.0104,  0.0167,  0.0120, -0.0068, -0.0423],
        [ 0.0072, -0.0253,  0.0072,  0.0072,  0.0023,  0.0253,  0.0253, -0.0041],
        [-0.02

In [ ]:
def quantize_scales_8bit(scales: torch.Tensor, block_size: int = 256):
    """
    Double quantization: compress the level-1 (per-64-block) NF4 scales
    down to 8-bit, using plain affine (uniform) quantization -- not NF4,
    since scale values are positive-only and don't follow a zero-centered
    Gaussian shape.
    """
    mean = scales.mean()
    centered = scales - mean  # remove the positive-only bias

    n = centered.numel()
    pad = (-n) % block_size
    if pad:
        centered = torch.cat([centered, torch.zeros(pad)])
    blocks = centered.view(-1, block_size)

    block_scales = blocks.abs().max(dim=1, keepdim=True).values.clamp(min=1e-8)
    normalized = blocks / block_scales  # roughly [-1, 1]

    # uniform 8-bit levels: 256 evenly spaced points from -1 to 1
    levels = torch.linspace(-1, 1, 256)
    diffs = (normalized.unsqueeze(-1) - levels.view(1, 1, -1)).abs()
    indices = diffs.argmin(dim=-1).to(torch.uint8)

    return indices, block_scales.squeeze(1), mean, n